<a href="https://colab.research.google.com/github/kempegowda7/AI/blob/main/unification_algorithem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re

def is_variable(t):
    """
    Defines variables. In standard logic, lowercase letters
    from u to z (u, v, w, x, y, z) are typically used as variables.
    """
    return isinstance(t, str) and t in ['x', 'y', 'z', 'w', 'v', 'u']

def is_atomic(t):
    """Returns True if the term is a single variable or a constant."""
    return isinstance(t, str)

def occurs_in(var, expr):
    """Checks if a variable occurs inside a compound expression (Occurs Check)."""
    if expr == var:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, arg) for arg in expr[1:])
    return False

def apply_subst(subst, expr):
    """Applies a substitution set to an expression."""
    if isinstance(expr, str):
        if expr in subst:
            return apply_subst(subst, subst[expr])
        return expr
    elif isinstance(expr, list):
        return [expr[0]] + [apply_subst(subst, arg) for arg in expr[1:]]
    return expr

def compose_subst(new_s, current_subst):
    """Composes two substitution sets together (APPEND operation)."""
    res = {}
    for var, val in current_subst.items():
        res[var] = apply_subst(new_s, val)
    for var, val in new_s.items():
        if var not in res:
            res[var] = val
    return res

def unify(psi1, psi2):
    """The complete Unification Algorithm matching the image steps."""

    # Step 1: If Psi1 or Psi2 is a variable or constant
    if is_atomic(psi1) or is_atomic(psi2):
        # a) If identical, return NIL (represented as empty dictionary {})
        if psi1 == psi2:
            return {}
        # b) Else if Psi1 is a variable
        elif is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            else:
                return {psi1: psi2}
        # c) Else if Psi2 is a variable
        elif is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}
        # d) Else return FAILURE (Two different constants)
        else:
            return "FAILURE"

    # Step 2: If initial Predicate symbols are not the same
    if psi1[0] != psi2[0]:
        return "FAILURE"

    # Step 3: If different number of arguments
    if len(psi1) != len(psi2):
        return "FAILURE"

    # Step 4: Set Substitution set (SUBST) to NIL
    SUBST = {}

    # Create working copies of the structures to prevent unwanted side-effects
    p1 = [psi1[0]] + list(psi1[1:])
    p2 = [psi2[0]] + list(psi2[1:])

    # Step 5: For i=1 to the number of arguments
    for i in range(1, len(p1)):
        # a) Call Unify with the ith elements
        S = unify(p1[i], p2[i])

        # b) If S = failure, then return Failure
        if S == "FAILURE":
            return "FAILURE"

        # c) If S is not NIL
        if S != {}:
            # a. Apply S to the remaining elements
            for j in range(i + 1, len(p1)):
                p1[j] = apply_subst(S, p1[j])
                p2[j] = apply_subst(S, p2[j])
            # b. SUBST = APPEND(S, SUBST)
            SUBST = compose_subst(S, SUBST)

    # Step 6: Return SUBST
    return SUBST

def parse_expression(s):
    """Parses user string input like 'P(x, a)' into nested list format ['P', 'x', 'a']"""
    s = s.replace(" ", "")
    match = re.match(r'^([A-Za-z0-9_]+)\((.*)\)$', s)
    if match:
        pred = match.group(1)
        args_str = match.group(2)
        args = []
        current = []
        depth = 0
        for char in args_str:
            if char == ',':
                if depth == 0:
                    args.append(parse_expression("".join(current)))
                    current = []
                else:
                    current.append(char)
            else:
                if char == '(':
                    depth += 1
                elif char == ')':
                    depth -= 1
                current.append(char)
        if current:
            args.append(parse_expression("".join(current)))
        return [pred] + args
    return s

def format_subst(subst):
    """Formats the substitution dictionary to match the standard standard {val/var} syntax."""
    if subst == "FAILURE":
        return "❌ FAILURE: Unification is NOT possible!"
    if subst == {}:
        return "✅ NIL (Expressions are already identical)"

    elements = [f"{val}/{key}" for key, val in subst.items()]
    return "✅ Success! Substitution Set: { " + ", ".join(elements) + " }"

# --- Main User Interface Line ---
print("=== Unification Algorithm Program ===")
print("Rules: Use lowercase letters (u, v, w, x, y, z) for variables.")
print("       Use other characters/words for predicates and constants.\n")

expr1_input = input("Enter Expression 1 (e.g., P(x, a)): ")
expr2_input = input("Enter Expression 2 (e.g., P(b, y)): ")

try:
    parsed1 = parse_expression(expr1_input)
    parsed2 = parse_expression(expr2_input)

    print(f"\nParsed Structure 1: {parsed1}")
    print(f"\nParsed Structure 2: {parsed2}")

    result = unify(parsed1, parsed2)

    print("\n--- RESULT ---")
    print(format_subst(result))

except Exception as e:
    print(f"\nAn error occurred while parsing input: {e}")


=== Unification Algorithm Program ===
Rules: Use lowercase letters (u, v, w, x, y, z) for variables.
       Use other characters/words for predicates and constants.

Enter Expression 1 (e.g., P(x, a)): p(A,B,B)
Enter Expression 2 (e.g., P(b, y)): p(x,y,z)

Parsed Structure 1: ['p', 'A', 'B', 'B']

Parsed Structure 2: ['p', 'x', 'y', 'z']

--- RESULT ---
✅ Success! Substitution Set: { A/x, B/y, B/z }
